In [ ]:
import os
import cv2
import pandas as pd
from pathlib import Path
from tqdm import tqdm
from omegaconf import OmegaConf

def prepare_data():
    cfg = OmegaConf.load("../config/training/trocr.yaml")
    output_dir = Path(cfg.data.processed_output_path) / "train"
    output_dir.mkdir(parents=True, exist_ok=True)
    
    root_img_path = Path(cfg.data.raw_images_path)
    metadata = []

    with open(cfg.data.raw_annotations_path, "r", encoding="utf-8") as f:
        # Пропускаем заголовок, если он есть
        lines = f.readlines()[1:] 

    for i, line in enumerate(tqdm(lines, desc="Processing crops")):
        parts = line.strip().split(",")
        if len(parts) < 6: continue
        
        img_rel_path, x1, y1, x2, y2, text = parts[0], int(parts[1]), int(parts[2]), int(parts[3]), int(parts[4]), parts[5]
        
        img_full_path = root_img_path / img_rel_path
        image = cv2.imread(str(img_full_path))
        if image is None: continue

        # Кроп с небольшим запасом (padding)
        h_img, w_img = image.shape[:2]
        pad = 3
        crop = image[max(0, y1-pad):min(h_img, y2+pad), max(0, x1-pad):min(w_img, x2+pad)]
        
        if crop.size == 0: continue

        # Имя файла делаем уникальным (одна картинка -> много номеров)
        file_name = f"crop_{i}.jpg"
        cv2.imwrite(str(output_dir / file_name), crop)
        metadata.append({"file_name": file_name, "text": text})

    pd.DataFrame(metadata).to_csv(output_dir / "metadata.csv", index=False)
    print(f"Готово! Кропов сохранено: {len(metadata)}")

if __name__ == "__main__":
    prepare_data()

In [1]:
import torch
from pathlib import Path
from omegaconf import OmegaConf
from datasets import load_dataset
from transformers import (
    TrOCRProcessor, VisionEncoderDecoderModel, 
    Seq2SeqTrainer, Seq2SeqTrainingArguments, default_data_collator
)
from torchvision import transforms

# Определяем пайплайн аугментаций
augmentation = transforms.Compose([
    # Небольшие повороты (очень важно для номеров под углом)
    transforms.RandomRotation(degrees=8, fill=255), 
    
    # Изменение яркости и контрастности (поможет при разном освещении)
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    
    # Небольшое размытие (симулирует не фокус или движение)
    transforms.GaussianBlur(kernel_size=(3, 3), sigma=(0.1, 1.0)),
    
    # Перспективные искажения (симулирует взгляд сбоку)
    transforms.RandomPerspective(distortion_scale=0.2, p=0.5, fill=255),
])

def train():
    cfg = OmegaConf.load("../config/training/trocr.yaml")
    
    # 1. Загрузка данных через ImageFolder
    data_path = Path(cfg.data.processed_output_path) / "train"
    dataset = load_dataset("imagefolder", data_dir=str(data_path), split="train")
    dataset = dataset.train_test_split(test_size=1 - cfg.data.train_split)

    # 2. Инициализация процессора и модели
    processor = TrOCRProcessor.from_pretrained(cfg.model.name)
    model = VisionEncoderDecoderModel.from_pretrained(cfg.model.name)

    # Настройка спец-токенов
    model.config.decoder_start_token_id = processor.tokenizer.cls_token_id
    model.config.pad_token_id = processor.tokenizer.pad_token_id
    model.config.vocab_size = model.config.decoder.vocab_size

    def preprocess(batch):
        # Применяем аугментации к списку картинок
        # batch["image"] — это список объектов PIL Image
        images = [augmentation(img.convert("RGB")) for img in batch["image"]]
        
        # Теперь скармливаем аугментированные картинки в процессор TrOCR
        inputs = processor(images=images, return_tensors="pt")
        
        labels = processor.tokenizer(batch["text"], 
                                     padding="max_length", 
                                     max_length=cfg.model.max_length).input_ids
        
        # Игнорируем паддинг в расчете лосса
        labels = [[(l if l != processor.tokenizer.pad_token_id else -100) for l in label] for label in labels]
        inputs["labels"] = torch.tensor(labels)
        return inputs

    train_ds = dataset["train"].with_transform(preprocess)
    eval_ds = dataset["test"].with_transform(preprocess)

    # 3. Настройка трейнера
    args = Seq2SeqTrainingArguments(
        output_dir=cfg.training.output_dir,
        predict_with_generate=True,
        eval_strategy="epoch",
        save_strategy="epoch",
        per_device_train_batch_size=cfg.training.batch_size,
        learning_rate=cfg.training.lr,
        num_train_epochs=cfg.training.epochs,
        logging_steps=10,
        load_best_model_at_end=True
    )

    trainer = Seq2SeqTrainer(
        model=model,
        args=args,
        train_dataset=train_ds,
        eval_dataset=eval_ds,
        data_collator=default_data_collator,
        tokenizer=processor,
    )

    trainer.train()
    model.save_pretrained("./final_model")
    processor.save_pretrained("./final_model")

if __name__ == "__main__":
    train()

Resolving data files:   0%|          | 0/4512 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/478 [00:00<?, ?it/s]

VisionEncoderDecoderModel LOAD REPORT from: microsoft/trocr-base-printed
Key                         | Status  | 
----------------------------+---------+-
encoder.pooler.dense.bias   | MISSING | 
encoder.pooler.dense.weight | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


TypeError: Seq2SeqTrainer.__init__() got an unexpected keyword argument 'tokenizer'